<a href="https://colab.research.google.com/github/1900690/OCR/blob/main/OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# パッケージのインストール（ArUco用のopencv-contrib-pythonを削除）
!pip install yomitoku nest_asyncio pandas lxml

# Colab環境での非同期処理エラーを回避するための設定
import nest_asyncio
nest_asyncio.apply()

import os
import cv2
import numpy as np
import pandas as pd
from google.colab.patches import cv2_imshow
from yomitoku import DocumentAnalyzer
from yomitoku.data.functions import load_pdf, load_image

# DocumentAnalyzerの初期化
analyzer = DocumentAnalyzer(visualize=True, device="cuda")

In [ ]:
# 対象ファイルのパス
file_path = "/content/SKM_C301i26051316520.png"

# ② 拡張子に応じた読み込み処理
ext = os.path.splitext(file_path)[1].lower()
if ext == '.pdf':
    imgs = load_pdf(file_path)
elif ext in ['.png', '.jpg', '.jpeg']:
    imgs = load_image(file_path)
else:
    raise ValueError(f"対応していない拡張子です: {ext}")

# 1ページ目の画像を取得
img = imgs[0]
if not isinstance(img, np.ndarray):
    img = np.array(img)

# ArUcoマーカー処理を削除したため、読み込んだ画像をそのまま使用
processed_img = img

# 解析を実行
results, ocr_vis, layout_vis = analyzer(processed_img)

# HTMLとして結果を出力
out_html_path = "/content/result.html"
html = results.to_html(out_html_path, img=processed_img)
with open(out_html_path, "w", encoding="utf-8") as f:
    f.write(html)

# ④ 出力されるhtmlファイルを自動的にCSVファイルに変換
try:
    # pandasを使用してHTML内の表をすべてDataFrameとして読み込む
    tables = pd.read_html(out_html_path)
    for i, df in enumerate(tables):
        csv_path = f"/content/result_table_{i}.csv"
        # utf-8-sigを指定することで、Excelで開いた際の文字化けを防止
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"CSV変換完了: {len(tables)}件の表を保存しました。")
except ValueError:
    print("HTML内に表が見つからなかったため、CSV変換は行われませんでした。")
except Exception as e:
    print(f"CSV変換中にエラーが発生しました: {e}")

# 可視化画像の保存
cv2.imwrite("/content/result_ocr.jpg", ocr_vis)
cv2.imwrite("/content/result_layout.jpg", layout_vis)

# ③ 出力画像のColab上での表示
print("【OCR検出結果】")
cv2_imshow(ocr_vis)

print("【レイアウト解析結果】")
cv2_imshow(layout_vis)